# 🧹 Plan de Preprocesado / Limpieza

## 1️⃣ Limpieza de columnas
- Eliminar `IsHomophobic` e `IsRadicalism` (sin ejemplos positivos)
- Target binario: `IsToxic` (True/False)

## 2️⃣ Limpieza del texto
1. Minúsculas
2. Eliminar URLs
3. Eliminar puntuación y números
4. Eliminar espacios extra

## 3️⃣ Filtrado de filas
- Eliminar comentarios con menos de 3 palabras
- Eliminar outliers extremos (revisar impacto antes de fijar el corte)

## 4️⃣ Stopwords + Lemmatización
1. Eliminar stopwords primero
2. Lematizar lo que queda

## 5️⃣ Vectorización
- Baseline con **TF-IDF**

In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


df = pd.read_csv('../../data/raw/youtoxic_english_1000.csv')
df. head()

,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\r\nDont you reckon them 'black lives matter' ...,True,True,False,False,True,False,False,False,False,False,False,False
3,Ugg7Gd006w1MPngCoAEC,04kJtp6pVXI,There are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,Ugg8FfTbbNF8IngCoAEC,04kJtp6pVXI,"The Arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,False,False


In [4]:
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords

# Ver palabras vacías en español
print(stopwords.words('spanish')[:5]) 
# Resultado: ['de', 'la', 'que', 'el', 'en']

# Buscar sinónimos de 'perro' usando OMW
syns = wn.synsets('perro', lang='spa')
print(syns[0].lemma_names('spa'))
# Resultado probable: ['can', 'perro', 'chucho...']

['de', 'la', 'que', 'el', 'en']
['can', 'perro']


### Limpieza de columnas

In [5]:
# Eliminar columnas sin ejemplos positivos
df.drop(columns=['IsHomophobic', 'IsRadicalism'], inplace=True)

# Para el modelo binario solo nos quedamos con IsToxic como target
# El resto de etiquetas las dejamos pero no las usaremos
target = df['IsToxic']

### Limpieza del teto 

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Text_clean'] = df['Text'].apply(clean_text)

### Vamos aplicar esta regla para saber cuantas se pierden

In [7]:
# Ver impacto del filtrado antes de aplicarlo
df['word_count'] = df['Text_clean'].apply(lambda x: len(x.split()))

print(f"Total filas: {len(df)}")
print(f"Filas con menos de 3 palabras: {(df['word_count'] < 3).sum()}")
print(f"Filas con más de 200 palabras: {(df['word_count'] > 200).sum()}")
print(f"Filas que quedarían: {((df['word_count'] >= 3) & (df['word_count'] <= 200)).sum()}")

Total filas: 1000
Filas con menos de 3 palabras: 39
Filas con más de 200 palabras: 11
Filas que quedarían: 950


### Aplicamos el Filtro

In [8]:
df = df[(df['word_count'] >= 3) & (df['word_count'] <= 200)].reset_index(drop=True)
print(f"Filas finales: {len(df)}")

Filas finales: 950


usamos pos='v' para que lematice como verbos, así "running" → "run" correctamente, que era el problema en el análisis del EDA.

In [9]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]  # quitar stopwords
    words = [lemmatizer.lemmatize(w, pos='v') for w in words]  # lematizar como verbos
    return ' '.join(words)

df['Text_final'] = df['Text_clean'].apply(preprocess_text)